In [ ]:
!pip install -q polars faiss-cpu

In [ ]:
import os
import gc
import shutil
import pandas as pd
import numpy as np
import polars as pl
import scipy.sparse as sp
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm.auto import tqdm
import traceback

# 1. Tối ưu phân mảnh VRAM
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# 2. Seed everything an toàn (Bao quát Multi-GPU trên Kaggle)
def seed_everything(seed=42):
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed) 
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Đang sử dụng thiết bị: {device}")
if torch.cuda.device_count() > 1:
    print(f"Phát hiện {torch.cuda.device_count()} GPUs. Đã sẵn sàng chạy Multi-GPU!")

In [ ]:
# ---------------------------------------------------------
# CẤU HÌNH ĐƯỜNG DẪN KAGGLE
# ---------------------------------------------------------
DATASET_DIR_NAME = 'datasets/shinnraa/data-train-amazon/processed'
INPUT_DIR        = f'/kaggle/input/{DATASET_DIR_NAME}'
WORKING_DIR      = '/kaggle/working'

# Đọc trực tiếp từ /kaggle/input 
TRAIN_PATH         = os.path.join(INPUT_DIR, 'train_interactions.parquet')
MODEL_SAVE_PATH    = os.path.join(WORKING_DIR, 'lightgcn_model.pth')
LIGHTGCN_CAND_PATH = os.path.join(WORKING_DIR, 'lightgcn_candidates.parquet')
CHUNK_DIR          = os.path.join(WORKING_DIR, 'lightgcn_chunks')

print("Đang đọc dữ liệu bằng Polars...")
# BƯỚC KHẮC PHỤC CHÍNH: Dùng .unique() để dọn sạch rác nhân bản 100%
df_train_pl = pl.read_parquet(TRAIN_PATH, columns=['mapped_user_id', 'mapped_item_id']).unique()
u_idx = df_train_pl['mapped_user_id'].to_numpy()
i_idx = df_train_pl['mapped_item_id'].to_numpy()

num_users = int(u_idx.max()) + 1
num_items = int(i_idx.max()) + 1
valid_user_ids = np.unique(u_idx)  # sorted array, chỉ gồm các ID có trong train
print(f"Số user thực sự có tương tác: {len(valid_user_ids):,} / {num_users:,}")
print(f"Tổng số Users (kể cả pad): {num_users:,}")
print(f"Tổng số Items (kể cả pad): {num_items:,}")

print("Đang xây dựng ma trận kề đồ thị (Adjacency Matrix)...")
adj = sp.coo_matrix((np.ones(len(u_idx)), (u_idx, i_idx + num_users)), shape=(num_users+num_items, num_users+num_items))
adj = adj + adj.T

# Chuẩn hóa D^{-1/2} A D^{-1/2}
d_inv = np.power(np.array(adj.sum(1)), -0.5).flatten()
d_inv[np.isinf(d_inv)] = 0.
d_mat = sp.diags(d_inv)

print("Đang nén đồ thị sang PyTorch COO chuẩn...")
coo = d_mat.dot(adj).dot(d_mat).tocoo()
indices = torch.tensor(np.vstack((coo.row, coo.col)), dtype=torch.long)
values = torch.tensor(coo.data, dtype=torch.float32)

norm_adj_t = torch.sparse_coo_tensor(indices, values, size=coo.shape).to(device)

# Dọn dẹp sạch sẽ RAM CPU
del adj, d_inv, d_mat, coo, df_train_pl, u_idx, i_idx
gc.collect()

In [ ]:
EMBED_DIM_LGCN = 64
BATCH_SIZE_LGCN = 20480
EPOCHS = 20

class LightGCN(nn.Module):
    def __init__(self, u, i, dim):
        super().__init__()
        self.u_emb = nn.Embedding(u, dim)
        self.i_emb = nn.Embedding(i, dim)
        nn.init.normal_(self.u_emb.weight, std=0.1)
        nn.init.normal_(self.i_emb.weight, std=0.1)
        
    def forward(self, adj):
        emb0 = torch.cat([self.u_emb.weight, self.i_emb.weight])
        e1 = torch.sparse.mm(adj, emb0)
        e2 = torch.sparse.mm(adj, e1)
        return torch.split((emb0 + e1 + e2) / 3.0, [num_users, num_items])

model_lgcn = LightGCN(num_users, num_items, EMBED_DIM_LGCN).to(device)

# if torch.cuda.device_count() > 1:
#     model_lgcn = nn.DataParallel(model_lgcn)

optimizer = torch.optim.Adam(model_lgcn.parameters(), lr=0.001)

print("Đang chuẩn bị Tensor trên RAM cho LightGCN...")
# Đọc lại cặp positive, nhớ phải áp dụng unique() y như lúc tạo đồ thị
pos_pairs = pl.read_parquet(TRAIN_PATH, columns=['mapped_user_id', 'mapped_item_id']).unique().to_numpy()
pos_pairs_tensor = torch.tensor(pos_pairs, dtype=torch.long)

model_lgcn.train()
for ep in range(EPOCHS):
    idx_perm = torch.randperm(len(pos_pairs_tensor))
    loss_ep, t_batches = 0, 0
    pbar = tqdm(range(0, len(pos_pairs_tensor), BATCH_SIZE_LGCN), desc=f"LightGCN Epoch {ep+1}/{EPOCHS}", leave=False)

    for i in pbar:
        b_idx = idx_perm[i:i+BATCH_SIZE_LGCN]
        batch = pos_pairs_tensor[b_idx].to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)

        u_reps, i_reps = model_lgcn(norm_adj_t)

        users = batch[:, 0]
        pos_items = batch[:, 1]
        neg_items = torch.randint(1, num_items, (len(batch),), device=device)

        u_emb = u_reps[users]
        pos_emb = i_reps[pos_items]
        neg_emb = i_reps[neg_items]

        # Lấy module base an toàn
        base_model = model_lgcn.module if hasattr(model_lgcn, 'module') else model_lgcn

        u_emb_0 = base_model.u_emb(users)
        pos_emb_0 = base_model.i_emb(pos_items)
        neg_emb_0 = base_model.i_emb(neg_items)

        del u_reps, i_reps # Rút củi VRAM

        pos_scores = (u_emb * pos_emb).sum(1)
        neg_scores = (u_emb * neg_emb).sum(1)
        bpr_loss = -F.logsigmoid(pos_scores - neg_scores).mean()

        reg_loss = (1/2) * (u_emb_0.norm(2).pow(2) + pos_emb_0.norm(2).pow(2) + neg_emb_0.norm(2).pow(2)) / float(len(users))
        loss = bpr_loss + 1e-4 * reg_loss

        del u_emb, pos_emb, neg_emb, pos_scores, neg_scores, u_emb_0, pos_emb_0, neg_emb_0

        loss.backward()
        optimizer.step()

        loss_ep += loss.item()
        t_batches += 1
        pbar.set_postfix(loss=f"{loss_ep/t_batches:.4f}")

    if (ep+1) % 5 == 0:
        print(f"LightGCN Epoch {ep+1:02d}/{EPOCHS} | Avg Loss: {loss_ep/t_batches:.4f}")

base_model = model_lgcn.module if hasattr(model_lgcn, 'module') else model_lgcn
torch.save(base_model.state_dict(), MODEL_SAVE_PATH)
print(f"✅ Đã lưu trọng số tại: {MODEL_SAVE_PATH} (Nằm trong thư mục /kaggle/working/)")

In [ ]:
torch.cuda.empty_cache()
gc.collect()

model_lgcn.eval()
base_model_lgcn = model_lgcn.module if hasattr(model_lgcn, 'module') else model_lgcn

os.makedirs(CHUNK_DIR, exist_ok=True)
INFER_BATCH_SIZE = 512
CHUNK_SIZE = 1000 
K_RANK = 100

print(f"Đang trích xuất Top {K_RANK} LightGCN (Batch size: {INFER_BATCH_SIZE})...")

users_list = []
items_list = []
chunk_idx = 0

with torch.no_grad():
    # Gọi đồ thị 1 lần duy nhất từ base_model
    u_e, i_e = base_model_lgcn(norm_adj_t) 

    pbar = tqdm(range(0, len(valid_user_ids), INFER_BATCH_SIZE), desc="Inference")
    for i in pbar:
        u_batch_ids = valid_user_ids[i : i + INFER_BATCH_SIZE]  # ← dùng valid_user_ids thay vì np.arange
    
        u_batch = u_e[u_batch_ids]
        scores = torch.matmul(u_batch, i_e.T)
        scores[:, 0] = float('-inf')
    
        _, top_idx = torch.topk(scores, K_RANK, dim=1)
    
        users_list.append(np.repeat(u_batch_ids, K_RANK).astype(np.int32))
        items_list.append(top_idx.cpu().numpy().flatten().astype(np.int32))
    
        del u_batch, scores, top_idx

        # Ghi chunk
        if len(users_list) >= CHUNK_SIZE or (i + INFER_BATCH_SIZE) >= len(valid_user_ids):
            final_u = np.concatenate(users_list)
            final_i = np.concatenate(items_list)
            
            df_chunk = pd.DataFrame({
                'mapped_user_id': final_u,
                'mapped_item_id': final_i
            })
            
            # Tính rank an toàn 100% (chiều dài luôn chia hết cho K_RANK)
            df_chunk['lightgcn_rank'] = np.tile(np.arange(1, K_RANK+1, dtype=np.int8), len(final_u) // K_RANK)
            
            chunk_path = f"{CHUNK_DIR}/chunk_{chunk_idx}.parquet"
            df_chunk.to_parquet(chunk_path)
            
            users_list, items_list = [], []
            chunk_idx += 1
            gc.collect()

del u_e, i_e
torch.cuda.empty_cache()
gc.collect()

# Gộp file an toàn
try:
    print("Đang gộp các file chunk bằng Polars...")
    lf_lightgcn = pl.scan_parquet(f'{CHUNK_DIR}/chunk_*.parquet')
    lf_lightgcn.sink_parquet(LIGHTGCN_CAND_PATH)
    
    shutil.rmtree(CHUNK_DIR)
    print(f"✅ Gộp thành công. Dọn rác xong. Kết quả hoàn chỉnh: {LIGHTGCN_CAND_PATH}")
    print("👉 Bạn có thể tải file này từ mục 'Output' ở cột bên phải của Kaggle!")

except Exception as e:
    print("❌ LỖI TRONG QUÁ TRÌNH GỘP PARQUET!")
    print(traceback.format_exc())
    print("Các file chunk tạm vẫn được giữ lại tại:", CHUNK_DIR)